In [ ]:
# Hoặc clone từ Hugging Face
!git lfs install
# !git clone https://huggingface.co/Devtamin/keyword-extraction-viet
!git clone https://github.com/NguyenTuan-UET/keyword-extraction-viet
%cd keyword-extraction-viet

In [ ]:
# Cell 2: Cài đặt đầy đủ dependencies
!pip install -q torch transformers
!pip install -q py_vncorenlp
!pip install -q scikit-learn
!pip install -q underthesea
!pip install -q numpy
!pip install -q gradio

print("✓ All dependencies installed!")

In [ ]:
# Cell 4: Tải và lưu models
import torch
from transformers import AutoModel, AutoModelForTokenClassification
import os

# Tạo thư mục
os.makedirs('pretrained-models', exist_ok=True)

# Tải PhoBERT
print("Đang tải PhoBERT...")
phobert = AutoModel.from_pretrained("vinai/phobert-base-v2")
phobert.eval()
torch.save(phobert, 'pretrained-models/phobert.pt')

# Tải NER model
print("Đang tải NER model...")
ner_model = AutoModelForTokenClassification.from_pretrained("NlpHUST/ner-vietnamese-electra-base")
ner_model.eval()
torch.save(ner_model, 'pretrained-models/ner-vietnamese-electra-base.pt')

print("Hoàn tất tải models!")

In [ ]:
import sys
%cd /content/keyword-extraction-viet
sys.path.append('/content/keyword-extraction-viet')

from pipeline import KeywordExtractorPipeline
import torch

# Load với weights_only=False
phobert = torch.load('pretrained-models/phobert.pt', weights_only=False)
phobert.eval()
ner_model = torch.load('pretrained-models/ner-vietnamese-electra-base.pt', weights_only=False)
ner_model.eval()

kw_pipeline = KeywordExtractorPipeline(phobert, ner_model)
print("✓ Pipeline ready!")

In [ ]:
title = ""
text = """
Trong kỳ điều chỉnh chiều ngày 5/3, giá xăng dầu đồng loạt tăng mạnh.
  Trong đó, giá xăng tăng 1.900-2.190 đồng/lít, giá dầu tăng 1.800-7.140 đồng/lít.
  Chiều 5/3, liên Bộ Công Thương - Tài chính điều chỉnh giá bán lẻ xăng dầu, áp dụng từ 15h cùng ngày.
  Sau điều chỉnh, giá bán lẻ với xăng E5 RON 92 là 21.440 đồng/lít và xăng RON 95 là 22.340 đồng/lít.
  Về số dư Quỹ bình ổn giá xăng dầu, đến hết quý III/2025, tổng số dư Quỹ bình ổn giá xăng dầu của 27 thương nhân đầu mối là 5.617 tỷ đồng, tăng gần 3 tỷ đồng so với quý trước đó.
  Về nguồn cung xăng dầu, nhiều doanh nghiệp đầu mối cho biết nguồn cung xăng dầu vẫn được đảm bảo.
  Doanh nghiệp cũng chủ động kế hoạch tạo nguồn năm 2026, ký hợp đồng với hai nhà máy lọc dầu trong nước và bố trí nhập khẩu theo kế hoạch.
  Riêng 10 ngày đầu tháng 3, sản lượng xăng dầu nhập khẩu tăng khoảng 50% so với mức bình quân tháng.

  """

In [ ]:
# @title keybert raw
!pip install -q keybert sentence-transformers pyvi

import requests
from keybert import KeyBERT

doc = """
During the adjustment period on the afternoon of March 5, gasoline and oil prices sharply increased across the board. Specifically, gasoline prices increased by 1,900-2,190 VND/liter, and oil prices increased by 1,800-7,140 VND/liter. On the afternoon of March 5, the inter-Ministry of Industry and Trade - Finance adjusted retail gasoline and oil prices, effective from 15:00 on the same day. Following the adjustment, the retail price for E5 RON 92 gasoline is 21,440 VND/liter and for RON 95 gasoline is 22,340 VND/liter. Regarding the balance of the Petroleum Price Stabilization Fund, by the end of Q3/2025, the total balance of the Petroleum Price Stabilization Fund of 27 key fuel traders was 5,617 billion VND, an increase of nearly 3 billion VND compared to the previous quarter. Regarding the petroleum supply, many focal enterprises stated that the petroleum supply remains ensured. Enterprises have also proactively planned their sourcing for 2026, signing contracts with two domestic oil refineries and arranging imports according to the plan. In the first 10 days of March alone, the imported petroleum volume increased by approximately 50% compared to the monthly average.
"""


In [ ]:
kw_model_raw = KeyBERT()
keywords_raw = kw_model_raw.extract_keywords (
    doc,
    keyphrase_ngram_range=(1, 3),
    use_mmr=True,
    diversity=0.35,
    top_n=10
)

print("OUTPUT:")
for word, score in keywords_raw:
    print(f"- {word.replace('_', ' ')} ({score:.4f})")

In [ ]:
kw_model_vi = KeyBERT()
keywords_raw_vi = kw_model_vi.extract_keywords (
    text,
    keyphrase_ngram_range=(1, 3),
    use_mmr=True,
    diversity=0.35,
    top_n=10
)

print("OUTPUT:")
for word, score in keywords_raw_vi:
    print(f"- {word.replace('_', ' ')} ({score:.4f})")

In [ ]:
# @title keybert_embedding_VietNam
!pip install -q keybert sentence-transformers pyvi

import requests
from keybert import KeyBERT
from pyvi import ViTokenizer
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'dangvantuan/vietnamese-embedding'

In [ ]:


def get_vietnamese_stopwords():
    url = "https://raw.githubusercontent.com/stopwords/vietnamese-stopwords/master/vietnamese-stopwords.txt"

    try:
        response = requests.get(url)
        raw_list = [line.strip() for line in response.text.split('\n') if line.strip()]

        # thay dấu cách bằng _
        final_list = [w.replace(" ", "_") for w in raw_list]

        return final_list

    except Exception as e:
        print(f"Lỗi: {e}")
        return []

# segmented
doc_segmented = ViTokenizer.tokenize(text)

print(f"\nVăn bản xử lý ({len(doc_segmented.split())} tokens):\n{doc_segmented[:200]}...\n")

embedding_model = SentenceTransformer(MODEL_NAME)
embedding_model.max_seq_length = 256
kw_model = KeyBERT(model=embedding_model)

vn_stopwords = get_vietnamese_stopwords()

keywords = kw_model.extract_keywords(
    doc_segmented,
    keyphrase_ngram_range=(1, 3),
    use_mmr=True,
    diversity=0.35,
    stop_words=vn_stopwords,
    top_n=10
)

print("OUTPUT:")
for word, score in keywords:
    print(f"- {word.replace('_', ' ')} ({score:.4f})")


In [ ]:
 # @title keybert VietNam
inp = {"title": title , "text": text}

keywords = kw_pipeline(
        inputs=inp,
        min_freq=1,
        ngram_n=(1, 3),
        top_n=10,
        diversify_result=False
    )

print("\nKeywords:")
for kw, score in keywords:
    print(f"  - {kw}: {score:.4f}")
print()